#### 常數

In [1]:
POSE_TYPE = "twist"
POSE_TAG = "close_30"

JSON_FILE_PATH = f"./json/{POSE_TYPE}/{POSE_TAG}.json"

RIGHT_OUTPUT_FOLDER_PATH = f"./output/draw_pose/{POSE_TYPE}/angle_right/{POSE_TAG}/"
FRONT_OUTPUT_FOLDER_PATH = f"./output/draw_pose/{POSE_TYPE}/angle_front/{POSE_TAG}/"
TOP_RIGHT_OUTPUT_FOLDER_PATH = f"./output/draw_pose/{POSE_TYPE}/angle_top_right/{POSE_TAG}/"
TOP_FRONT_OUTPUT_FOLDER_PATH = f"./output/draw_pose/{POSE_TYPE}/angle_top_front/{POSE_TAG}/"

#### 套件

In [2]:
from src.json_to_pose.base import Pose, PoseAnalyzer
from src.json_to_pose.values import blazepose_lines
from src.utils.process_data import transYAxis, scaleYAxis
import src.utils.plot_painter as plot_painter

import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# 參考類型
from mpl_toolkits.mplot3d import Axes3D

#### 變數

##### 參考視角參數

In [3]:
### 預設
# elev, azim, roll = 20, 30, 0

## 正面影像
# elev, azim, roll = 30, 90, 0  # 正面
# elev, azim, roll = 30, 180, 0  # 右側

## 右側影像
# elev, azim, roll = 30, 0, 0  # 正面
# elev, azim, roll = 30, 270, 0  # 右側

##### 圖表用變數

In [4]:
# 顯示資料範圍
data_range_x = [-1, 1]  # x 軸
data_range_y = [-1, 1]  # y 軸
data_range_z = [0, 2]  # z 軸

# 圖表資訊顏色
c_left = "#f00"  # 左邊
c_center = "#0f0"  # 右邊
c_right = "#00f"  # 中間
c_dot = "#000"  # 關鍵點

is_3d = True  # 圖表是否為 3D
dot_size = 5  # 繪製關鍵點大小
line_lst = blazepose_lines  # 設定連線列表

##### 資料物件

In [5]:
pose = Pose(JSON_FILE_PATH)
analyzer = PoseAnalyzer(pose)

right_out_path = Path(RIGHT_OUTPUT_FOLDER_PATH)
front_out_path = Path(FRONT_OUTPUT_FOLDER_PATH)
top_right_out_path = Path(TOP_RIGHT_OUTPUT_FOLDER_PATH)
top_front_out_path = Path(TOP_FRONT_OUTPUT_FOLDER_PATH)

##### 所需資料

In [6]:
pose_kpt_positions = pose.get_all_pose_kpt_positions(is_3d)
line_positions = analyzer.get_all_line_positions(blazepose_lines, is_3d)
labels = analyzer.get_all_lhc_labels(is_3d)

#### 函式

In [7]:
def save_3d_pose_result(
    view_init: list[float, float, float], output_file_prefix: str
) -> None:
    """儲存為 3D 姿勢結果

    Args:
        view_init (list[float, float, float]): 設定圖表視角
        output_file_prefix (str): 輸出檔案前綴名稱
    """
    elev, azim, roll = view_init
    for i in range(pose.get_image_count()):

        fig = plt.figure(figsize=(4.8, 6.4))  # 建立圖表基底圖片
        title = ""  # 標題名稱

        # 建立圖表座標(2D 或 3D)
        if not is_3d:
            ax = fig.add_subplot()
        else:
            ax: Axes3D = fig.add_subplot(projection="3d")
            ax.view_init(elev, azim, roll)

        if pose.get_pose_count(i) > 0:  # 如果圖片中有姿勢
            title = (
                f"label: {analyzer.get_pose_lhc_label(i, 0, is_3d)}\n"
                + f"staggered angle(3d): {analyzer.get_pose_shoulder_hip_staggered_angle(i, 0, is_3d):.2f}\n"
                # + f"staggered angle(xz): {analyzer.get_pose_shoulder_hip_staggered_angle_xz(i, 0):.2f}"
            )

            # 取得資料
            p_dot = pose_kpt_positions[i]
            p_left, p_center, p_right = line_positions[i]

            # 將座標進行反轉和平移
            add_y = np.array(p_dot)[:, 1].max()  # 取得要平移的 y 軸數值
            p_left = transYAxis(scaleYAxis(p_left, -1), add_y)
            p_center = transYAxis(scaleYAxis(p_center, -1), add_y)
            p_right = transYAxis(scaleYAxis(p_right, -1), add_y)
            p_dot = transYAxis(scaleYAxis(p_dot, -1), add_y)

            # 圖表繪製
            plot_painter.draw_lines(p_left, is_3d, c_left, ax)
            plot_painter.draw_lines(p_center, is_3d, c_center, ax)
            plot_painter.draw_lines(p_right, is_3d, c_right, ax)
            plot_painter.draw_dots(p_dot, is_3d, dot_size, c_dot, "keypoints", ax)
        else:  # 如果圖片中沒有姿勢
            title = "No Pose"

        # 圖表設定
        ax.set_title(title)
        plot_painter.set_data_range(data_range_x, data_range_y, data_range_z, ax)

        # 儲存圖表
        plt.savefig(f"{output_file_prefix}({i:03d}).png")
        plt.close()  # 關閉圖表

In [8]:
def save_3d_center_line_result(
    view_init: list[float, float, float], output_file_prefix: str
) -> None:
    """儲存為 3D 中心線條(肩膀和腰部)結果

    Args:
        view_init (list[float, float, float]): 設定圖表視角
        output_file_prefix (str): 輸出檔案前綴名稱
    """
    elev, azim, roll = view_init
    for i in range(pose.get_image_count()):

        fig = plt.figure(figsize=(4.8, 6.4))  # 建立圖表基底圖片
        title = ""  # 標題名稱

        # 建立圖表座標(2D 或 3D)
        if not is_3d:
            ax = fig.add_subplot()
            ax.set_xlabel("x")
            ax.set_ylabel("y")
        else:
            ax: Axes3D = fig.add_subplot(projection="3d")
            ax.view_init(elev, azim, roll)
            ax.set_xlabel("x")
            ax.set_ylabel("z")
            ax.set_zlabel("y")

        if pose.get_pose_count(i) > 0:  # 如果圖片中有姿勢
            title = (
                f"label: {analyzer.get_pose_lhc_label(i, 0, is_3d)}\n"
                + f"staggered angle(3d): {analyzer.get_pose_shoulder_hip_staggered_angle(i, 0, is_3d):.2f}\n"
                + f"staggered angle(xz): {analyzer.get_pose_shoulder_hip_staggered_angle_xz(i, 0):.2f}"
            )

            # 取得資料
            p_dot = pose_kpt_positions[i]
            _, p_center, _ = line_positions[i]

            # 將座標進行反轉和平移
            add_y = np.array(p_dot)[:, 1].max()  # 取得要平移的 y 軸數值
            p_center = transYAxis(scaleYAxis(p_center, -1), add_y)

            # 設定肩膀和腰部線條資訊
            shoulder_line = np.array(p_center[1])
            hip_line = np.array(p_center[2])

            # 圖表繪製
            ax.plot(
                shoulder_line[:, 0],
                shoulder_line[:, 2],
                shoulder_line[:, 1],
                label="shoulder",
            )
            ax.plot(hip_line[:, 0], hip_line[:, 2], hip_line[:, 1], label="hip")
            ax.legend()
        else:  # 如果圖片中沒有姿勢
            title = "No Pose"

        # 圖表設定
        ax.set_title(title)
        plot_painter.set_data_range(data_range_x, data_range_y, data_range_z, ax)

        # 儲存圖表
        plt.savefig(f"{output_file_prefix}({i:03d}).png")
        plt.close()  # 關閉圖表

#### 主程式

##### 建立資料夾

In [9]:
if not right_out_path.exists():
    right_out_path.mkdir(parents=True)
if not front_out_path.exists():
    front_out_path.mkdir(parents=True)
if not top_right_out_path.exists():
    top_right_out_path.mkdir(parents=True)
if not top_front_out_path.exists():
    top_front_out_path.mkdir(parents=True)

##### 右邊圖片輸出

In [10]:
elev, azim, roll = 30, 270, 0  # 右邊視角
output_file_prefix = right_out_path / POSE_TAG  # 輸出檔案前綴

In [11]:
save_3d_pose_result([elev, azim, roll], output_file_prefix)
print("Finish")

Finish


##### 正面圖片輸出

In [12]:
elev, azim, roll = 30, 0, 0  # 正面視角
output_file_prefix = front_out_path / POSE_TAG  # 輸出檔案前綴

In [13]:
save_3d_pose_result([elev, azim, roll], output_file_prefix)
print("Finish")

Finish


##### 上右方圖片輸出

In [14]:
elev, azim, roll = 90, 270, 0  # 右邊視角
output_file_prefix = top_right_out_path / POSE_TAG  # 輸出檔案前綴

In [15]:
save_3d_pose_result([elev, azim, roll], output_file_prefix)
print("Finish")

Finish


##### 上正前方圖片輸出

In [16]:
elev, azim, roll = 90, 0, 0  # 上正前方視角
output_file_prefix = top_front_out_path / POSE_TAG  # 輸出檔案前綴

In [17]:
save_3d_pose_result([elev, azim, roll], output_file_prefix)
print("Finish")

Finish


# 全部完成

In [18]:
print("Finish")

Finish
